# 0. les biblios

In [32]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Charger erp_business_partners.csv

In [33]:
# Charger les données
df = pd.read_csv("erp_facilities.csv")

In [34]:
# 1. Dimensions et types de colonnes
print("Dimensions :", df.shape)
print("\nTypes de colonnes :")
df.info()

Dimensions : (30, 11)

Types de colonnes :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   facility_code           30 non-null     object 
 1   facility_name           30 non-null     object 
 2   facility_category       30 non-null     object 
 3   operating_partner_code  30 non-null     object 
 4   street_address          30 non-null     object 
 5   region                  30 non-null     object 
 6   latitude                30 non-null     float64
 7   longitude               30 non-null     float64
 8   opening_date            30 non-null     object 
 9   facility_status         30 non-null     object 
 10  storage_capabilities    29 non-null     object 
dtypes: float64(2), object(9)
memory usage: 2.7+ KB


In [35]:
# 2. Échantillon des données
print("\nPremières lignes :")
display(df.head())


Premières lignes :


,facility_code,facility_name,facility_category,operating_partner_code,street_address,region,latitude,longitude,opening_date,facility_status,storage_capabilities
0,FCL-1011,Meridian Valenne Headquarters,office,BP-30078,"Meridian Valenne Headquarters, Valenne Central...",Valenne Central District,47.24,7.58,2006-01-01,active,ambient
1,FCL-1024,Meridian Kintara Biological Aggregation Facility,warehouse,BP-30078,Meridian Kintara Biological Aggregation Facili...,Kintara Karst Cluster,4.74,112.28,1997-01-01,active,ambient;secure_cage;2-8C;frozen
2,FCL-1019,Meridian Avelora Transfer Annex,transfer_hub,BP-30078,"Meridian Avelora Transfer Annex, Eastern Avelo...",Eastern Avelora Agricultural Cluster,7.46,101.91,2003-01-01,active,ambient;secure_cage;2-8C;frozen
3,FCL-1020,Meridian Tembara Coastal Hub,transfer_hub,BP-30078,"Meridian Tembara Coastal Hub, Tembara Coastal ...",Tembara Coastal Cluster,2.20,104.55,2010-01-01,active,ambient;secure_cage;2-8C;frozen
4,FCL-1016,Northstar Tembara Cold-Chain Hub,warehouse,BP-30047,"Northstar Tembara Cold-Chain Hub, Tembara Coas...",Tembara Coastal Cluster,2.23,104.58,2008-01-01,active,ambient;secure_cage;2-8C;frozen


In [36]:
# 3. Doublons
print("Nombre de lignes dupliquées (strictement identiques) :", df.duplicated().sum())
print("Nombre de facility_code dupliqués :", df['facility_code'].duplicated().sum())

# Afficher les doublons de facility_code s'il y en a
dupes = df[df['facility_code'].duplicated(keep=False)]
if not dupes.empty:
    display(dupes.sort_values('facility_code'))

Nombre de lignes dupliquées (strictement identiques) : 0
Nombre de facility_code dupliqués : 0


# 2. Valeurs manquantes

### a. Constat du problème

In [37]:
# 4. Valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

print("\nPourcentage de manquants par colonne :")
print((df.isna().sum() / len(df) * 100).round(2))

Valeurs manquantes par colonne :
facility_code             0
facility_name             0
facility_category         0
operating_partner_code    0
street_address            0
region                    0
latitude                  0
longitude                 0
opening_date              0
facility_status           0
storage_capabilities      1
dtype: int64

Pourcentage de manquants par colonne :
facility_code             0.00
facility_name             0.00
facility_category         0.00
operating_partner_code    0.00
street_address            0.00
region                    0.00
latitude                  0.00
longitude                 0.00
opening_date              0.00
facility_status           0.00
storage_capabilities      3.33
dtype: float64


In [38]:
# Voir les lignes où storage_capabilities est manquant
missing_capabilities = df[df['storage_capabilities'].isna()]
display(missing_capabilities)

,facility_code,facility_name,facility_category,operating_partner_code,street_address,region,latitude,longitude,opening_date,facility_status,storage_capabilities
8,FCL-1029,Vale Strategy Valenne Office,office,BP-30044,"Vale Strategy Valenne Office, Valenne Central ...",Valenne Central District,47.26,7.49,2009-01-01,active,NaN


In [39]:
# Supprimer les lignes où 'storage_capabilities' est NaN
df = df.dropna(subset=['storage_capabilities'])

print(df['storage_capabilities'].value_counts().head())

storage_capabilities
ambient;secure_cage                9
ambient;secure_cage;2-8C;frozen    8
ambient                            6
2-8C;controlled_access             6
Name: count, dtype: int64


il y a des doublons ici dans la colonne storage_capabilities

In [40]:
# Uniformiser l'ordre des capacités pour fusionner les doublons logiques
df['storage_capabilities'] = df['storage_capabilities'].astype(str).apply(
    lambda x: ';'.join(sorted([item.strip() for item in x.split(';')]))
)

# Vérifier le nouveau compte des valeurs uniques après nettoyage
print(df['storage_capabilities'].value_counts())

storage_capabilities
ambient;secure_cage                9
2-8C;ambient;frozen;secure_cage    8
ambient                            6
2-8C;controlled_access             6
Name: count, dtype: int64


j'ai fait ca pour la suite (one-hot encoding)

# 3. Doublons

### a- Doublons de lignes

In [41]:
print("Nombre de lignes dupliquées :", df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values('facility_code'))

Nombre de lignes dupliquées : 0


,facility_code,facility_name,facility_category,operating_partner_code,street_address,region,latitude,longitude,opening_date,facility_status,storage_capabilities


### b- Doublons de colonnes 

In [42]:
# 1. Vérifier si des noms de colonnes sont strictement identiques
noms_doublons = df.columns[df.columns.duplicated()]
print("Noms de colonnes en double :", noms_doublons.tolist())

# 2. Vérifier si le contenu de certaines colonnes est 100% identique
contenu_doublons = df.columns[df.T.duplicated()]
print("Colonnes avec un contenu dupliqué :", contenu_doublons.tolist())

Noms de colonnes en double : []
Colonnes avec un contenu dupliqué : []


# 4. Formats incohérents

### Les types de chaque colonnes :

In [43]:
print(df.dtypes)

facility_code              object
facility_name              object
facility_category          object
operating_partner_code     object
street_address             object
region                     object
latitude                  float64
longitude                 float64
opening_date               object
facility_status            object
storage_capabilities       object
dtype: object


### - Vérifier le format de facility_code

In [44]:
pattern_facility = r'^FCL-\d+$'
non_conformes_facility = df[~df['facility_code'].astype(str).str.match(pattern_facility, na=False)]
print(f"facility_code non conformes : {len(non_conformes_facility)}")

facility_code non conformes : 0


### - Vérifier le format de facility_name

In [45]:
print(df['facility_name'].value_counts(dropna=False))

facility_name
Meridian Valenne Headquarters                       1
Meridian Kintara Biological Aggregation Facility    1
Meridian Avelora Transfer Annex                     1
Meridian Tembara Coastal Hub                        1
Northstar Tembara Cold-Chain Hub                    1
Asterion Central Clinical Campus                    1
Asterion Avelora Executive Clinic                   1
Kintara Cave Authority Field Depot                  1
Threshold Earth Avelora Laboratory                  1
Valenne Central Distribution Centre                 1
Nordhaven Freight Exchange Warehouse                1
Civic Health Receiving Centre                       1
Auric Medical Supply Depot                          1
Tembara Port Consolidation Yard                     1
Kintara Field Equipment Depot                       1
Avelora Orchard Export Packhouse                    1
Nordhaven Refrigeration Workshop                    1
Valenne Laboratory Materials Store                  1
Tembara Coasta

### - Vérifier le format facility_category (catégorielles)

In [46]:
print(df['facility_category'].value_counts(dropna=False))

facility_category
warehouse                    12
field_depot                   4
laboratory_delivery_point     4
transfer_hub                  3
clinical_delivery_point       2
office                        1
port_staging                  1
service_facility              1
air_cargo_terminal            1
Name: count, dtype: int64


### - Vérifier le format operating_partner_code

In [47]:
pattern_partner = r'^BP-\d+$'

# Lignes non conformes à la regex
non_conformes_partner = df[~df['operating_partner_code'].astype(str).str.match(pattern_partner, na=False)]
print(f"operating_partner_code non conformes : {len(non_conformes_partner)}")

if len(non_conformes_partner) > 0:
    display(non_conformes_partner[['facility_code', 'operating_partner_code']])

operating_partner_code non conformes : 0


### - Vérifier le format street_address

In [48]:
# Vérifier si certaines adresses sont vides ou uniquement des espaces
adresses_vides = df[df['street_address'].astype(str).str.strip() == '']
print(f"\nstreet_address vides ou composées d'espaces : {len(adresses_vides)}")

# Détecter la présence éventuelle d'espaces multiples consécutifs
espaces_multiples = df[df['street_address'].astype(str).str.contains(r'\s{2,}', regex=True, na=False)]
print(f"street_address avec des espaces doubles/multiples : {len(espaces_multiples)}")

# Aperçu d'échantillons pour contrôle visuel
print("\nÉchantillon des adresses :")
display(df[['facility_code', 'street_address']].head())


street_address vides ou composées d'espaces : 0
street_address avec des espaces doubles/multiples : 0

Échantillon des adresses :


,facility_code,street_address
0,FCL-1011,"Meridian Valenne Headquarters, Valenne Central..."
1,FCL-1024,Meridian Kintara Biological Aggregation Facili...
2,FCL-1019,"Meridian Avelora Transfer Annex, Eastern Avelo..."
3,FCL-1020,"Meridian Tembara Coastal Hub, Tembara Coastal ..."
4,FCL-1016,"Northstar Tembara Cold-Chain Hub, Tembara Coas..."


### - Vérifier le format region (catégorielles)

In [49]:
print(df['region'].value_counts(dropna=False))

region
Valenne Central District                7
Tembara Coastal Cluster                 7
Kintara Karst Cluster                   6
Eastern Avelora Agricultural Cluster    6
Nordhaven Maritime District             3
Name: count, dtype: int64


### - Vérifier le format latitude

In [50]:
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
lat_invalides = df[(df['latitude'].isna()) | (df['latitude'] < -90) | (df['latitude'] > 90)]
print(f"Latitude invalides ou manquantes : {len(lat_invalides)}")

Latitude invalides ou manquantes : 0


### - Vérifier le format longitude

In [51]:
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
lon_invalides = df[(df['longitude'].isna()) | (df['longitude'] < -180) | (df['longitude'] > 180)]
print(f"Longitude invalides ou manquantes : {len(lon_invalides)}")

Longitude invalides ou manquantes : 0


### - Vérifier le format opening_date

In [52]:
pattern_date = r'^\d{4}-\d{2}-\d{2}$'
non_conformes_date = df[~df['opening_date'].astype(str).str.match(pattern_date, na=False)]
print("\nopening_date au format non conforme :", len(non_conformes_date))
# Afficher la ligne avec le format de date non conforme
display(non_conformes_date)


opening_date au format non conforme : 0


,facility_code,facility_name,facility_category,operating_partner_code,street_address,region,latitude,longitude,opening_date,facility_status,storage_capabilities


### - Vérifier le format facility_status

In [53]:
print(df['risk_class'].value_counts(dropna=False))

KeyError: 'risk_class'

### - Vérifier le format storage_capabilities

In [ ]:
print(df['storage_capabilities'].value_counts())

storage_capabilities
ambient;secure_cage                9
2-8C;ambient;frozen;secure_cage    8
ambient                            6
2-8C;controlled_access             6
Name: count, dtype: int64


# 5. Télécharger le csv nettoyé

In [54]:
df.to_csv('erp_facilities_cleaned.csv', index=False)